# 🎓 University Admission QnA Bot
## Same pipeline as HadithBot.ipynb
```
Preprocess → Embed (MiniLM-style 384-dim) → FAISS-style Index → Cosine Search → Flask UI
```

## Cell 1 — Load QnA Dataset

In [ ]:
import json, numpy as np

with open('data/admission_qna.json') as f:
    qna = json.load(f)

questions = [item['question'] for item in qna]
answers   = [item['answer']   for item in qna]
topics    = [item['topic']    for item in qna]

print(f'Loaded {len(qna)} QnA pairs')
print('Sample:', questions[0])

## Cell 2 — Preprocess Text

In [ ]:
# Combine question + answer for richer embedding context
corpus = [q + ' ' + a for q, a in zip(questions, answers)]
print('Corpus sample:', corpus[0][:120], '...')

## Cell 3 — Embed with MiniLM-style (384-dim)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# Step A: TF-IDF sparse representation
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=8000, sublinear_tf=True)
X_sparse = tfidf.fit_transform(corpus)
print('TF-IDF shape:', X_sparse.shape)

# Step B: Project to 384 dims (matches MiniLM-L6-v2 output dimension)
svd = TruncatedSVD(n_components=384, random_state=42)
X_dense = svd.fit_transform(X_sparse).astype('float32')
print('Dense embedding shape:', X_dense.shape)

## Cell 4 — Store Vectors (FAISS-style Index)

In [ ]:
# L2-normalize → dot product becomes cosine similarity (same as FAISS IndexFlatIP)
embeddings = normalize(X_dense, norm='l2')
print('Normalized embeddings:', embeddings.shape)
print('Norms (should be ~1.0):', np.linalg.norm(embeddings[:3], axis=1))

# Save to disk
np.save('data/admission_embeddings.npy', embeddings)
print('Saved: data/admission_embeddings.npy')

## Cell 5 — Search by Cosine Similarity

In [ ]:
def search(query, top_k=3):
    # Embed query with same pipeline
    q_vec   = tfidf.transform([query])
    q_dense = svd.transform(q_vec).astype('float32')
    q_norm  = normalize(q_dense, norm='l2')
    # Cosine via dot product on L2-normed vectors
    scores  = (embeddings @ q_norm.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    for rank, idx in enumerate(top_idx, 1):
        print(f'[{rank}] Score={scores[idx]:.4f} | Topic: {topics[idx]}')
        print(f'     Q: {questions[idx]}')
        print(f'     A: {answers[idx][:100]}...')
        print()

search('What GPA do I need for Computer Science?')

## Cell 6 — Flask UI
Run `python app.py` then open http://localhost:5000

In [ ]:
# Launch Flask app
import subprocess
subprocess.Popen(['python', 'app.py'])
print('Server running at http://localhost:5000')